# 🚀 Mission 4: Draft Day!

This is the mission everything else was building toward. Today you build a
**ranking machine**: it guesses how every player will do THIS season, works
out who's truly valuable, and prints your own cheat sheet for draft day.

⏱️ *About 30 minutes — worth every second.*

In [ ]:
import ffkit as ff
from ffkit import ____

players = ff.load_players()
schedules = ff.load_schedules()

rules = ff.rules("NFL Standard + IDP")   # ⭐ or any preset from Mission 3
pool = ff.build_player_pool(players, schedules, rules)
print(f"Draft pool ready: {pool['player_id'].nunique():,} draftable players 🏈")

## 🔮 Step 1: The crystal ball (projections)

Nobody knows the future — but the past gives clues. Say Bijan averaged 20
points per game last season and 15 the season before. What will he do this
year? **It depends on how much you trust each clue.** That's YOUR call, with
three knobs:

| knob | 0 means... | 1 means... |
|---|---|---|
| `past_matters` | only last season counts | every season counts equally |
| `durability` | assume everyone plays all 17 games | expect missed games to happen again |
| `steady_bonus` | *(0 = just use averages)* | +1 loves steady players, **-1** loves boom-or-bust |

There is no "correct" setting — this is where YOUR football brain comes in.

In [ ]:
projections = ff.make_projections(
    pool,
    past_matters=0.5,   # half-trust older seasons
    durability=0.5,     # worry a medium amount about missed games
    steady_bonus=0.0,   # no taste adjustment... yet
)
projections.sort_values("proj_points", ascending=False).head(10)

## 🧠 Step 2: The draft secret (value over replacement)

Look at your projections: the top QB probably out-scores the top RB. So why
do smart drafters take running backs first?!

**Because value = how much better a player is than the FREE replacement.**

Say every team already started a QB and the best *undrafted* QB would still
score 280 — but the best undrafted RB only scores 150. Then a 380-point QB is
really worth 380 − 280 = **100**, while a 280-point RB is worth 280 − 150 =
**130**. The RB wins, even with fewer points!

That difference is called **VOR** (Value Over Replacement) — the `value`
column below. It's the single smartest number on your whole cheat sheet:

In [ ]:
board = ff.big_board(projections, rules)
board.head(15)

## 🪄 MAGIC: your Big Board, as a chart

In [ ]:
ff.viz.big_board_chart(board, top=25)

In [ ]:
# ⭐ TRY IT: what happens when ONLY last season matters?
# Set past_matters to 0.0 and see who jumps up or crashes down.
# Then try 1.0 (all seasons equal). Watch the board reshuffle!
test_projections = ff.make_projections(pool, past_matters=____)
ff.big_board(test_projections, rules).head(10)

In [ ]:
# 🔑 ANSWER — peek only if you're stuck! (remove the # marks to run it)
# test_projections = ff.make_projections(pool, past_matters=0.0)
# ff.big_board(test_projections, rules).head(10)

## 🪄 MAGIC: steady stars vs. boom-or-bust

Every dot is a player. **Right = scores more.** **Higher = wilder weeks.**
So the bottom-right corner is where the steady superstars live. Hover to see
who's who!

In [ ]:
ff.viz.value_scatter(board)

## 🎛️ The Draft Machine

All three knobs as sliders. Move one — the whole board re-ranks before your
eyes. Argue with your siblings about the right settings. That's the point. 😄

In [ ]:
ff.widgets.draft_dashboard(pool, rules)

## 🧑‍⚖️ Step 3: You vs. the experts

Real fantasy experts publish draft rankings. The lab downloaded today's
expert consensus — let's see where your math **disagrees** with them:

In [ ]:
experts = ff.load_rankings()
compared = ff.compare_with_experts(board, experts)

sleepers = compared.dropna(subset=["we_disagree_by"]).nlargest(6, "we_disagree_by")
print("😴 SLEEPERS — players YOUR math likes way more than the experts do:")
sleepers[["player_display_name", "pos", "pos_rank", "expert_rank", "we_disagree_by"]]

In [ ]:
ff.viz.experts_scatter(compared)

**Who should you believe?** Honestly — both. Your math only knows the past.
Experts also know the *news*: injuries healing, trades, coaching changes, and
**rookies** (first-year players have no NFL stats, so your board can't see
them at all — check the expert list for those!). The best drafters use math
AND news.

## 📄 Step 4: Print your cheat sheet

In [ ]:
cheat_sheet = board.head(150).drop(columns=["headshot_url", "player_id"])
cheat_sheet.to_csv("my_draft_cheat_sheet.csv", index=False)
print("Saved my_draft_cheat_sheet.csv — open it, print it, win your draft! 🏆")

In [ ]:
# ⭐ FINAL CHALLENGE: build a Big Board for a totally different league style!
# Presets: 'NFL Standard + IDP', 'ESPN Full PPR', 'Half PPR',
#          'Standard + Team Defense'
other_rules = ff.rules(____)
other_pool = ff.build_player_pool(players, schedules, other_rules)
other_board = ff.big_board(ff.make_projections(other_pool), other_rules)
other_board.head(15)

In [ ]:
# 🔑 ANSWER — peek only if you're stuck! (remove the # marks to run it)
# other_rules = ff.rules("ESPN Full PPR")
# other_pool = ff.build_player_pool(players, schedules, other_rules)
# other_board = ff.big_board(ff.make_projections(other_pool), other_rules)
# other_board.head(15)

Compare that board to your league's board. Different rules → different top
picks. **That's why you never copy a random internet ranking** — it might be
for a league with different rules than yours!

## 🎉 Mission complete — you built a draft machine!

Quick draft-day wisdom from the data:
- 💎 Draft **value**, not just points (that's your VOR column)
- 🏃 The drop-off from great RBs to okay RBs is a cliff — mind the tiers
- 🦵 Never draft a kicker early (check where kickers sit on your board 😅)
- 📰 Check expert lists for rookies your math can't see

**Next → Mission 5: Set Your Lineup** — because after the draft, you have to
win every single week.